# MN-BERT 4-Class Mongolian Comment Classification
Fine-tuning `tugstugi/bert-base-mongolian-cased` on 4-class comment classification.

## 1. Installs & Imports

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas torch

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2. Load CSV & Explore

In [ ]:
df = pd.read_csv("./10k_labeled.csv")
print(f"Shape: {df.shape}")
print("\nLabel distribution per split:")
print(df.groupby("split")["label4_class"].value_counts().unstack(fill_value=0))
print("\n3 sample rows:")
df.head(3)

## 3. Label Mapping

In [ ]:
LABEL2ID = {"CONSTRUCTIVE": 0, "NEUTRAL": 1, "POSITIVE": 2, "TOXIC": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

df["label"] = df["label4_class"].map(LABEL2ID)
assert df["label"].notna().all(), "Unmapped labels found!"

print("Label mapping:", LABEL2ID)

## 4. Build HuggingFace Datasets & Tokenize

In [ ]:
MODEL_NAME = "tugstugi/bert-base-mongolian-cased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ds_dict = DatasetDict({
    split: Dataset.from_pandas(
        df[df["split"] == split][["text_ml_clean", "label"]].reset_index(drop=True)
    )
    for split in ["train", "val", "test"]
})

def tokenize_fn(batch):
    return tokenizer(
        batch["text_ml_clean"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

ds_dict = ds_dict.map(tokenize_fn, batched=True, batch_size=512)
ds_dict = ds_dict.rename_column("label", "labels")
ds_dict.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

for split in ds_dict:
    print(f"{split}: {len(ds_dict[split])} samples")

## 5. Model & Class-Weighted Trainer

In [ ]:
# Compute class weights from training set
train_labels = df[df["split"] == "train"]["label"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_LABELS),
    y=train_labels,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", {ID2LABEL[i]: f"{w:.4f}" for i, w in enumerate(class_weights)})

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

## 6. Training Arguments

In [ ]:
CONFIG = {
    "num_train_epochs": 5,
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_macro",
    "greater_is_better": True,
    "fp16": torch.cuda.is_available(),
    "gradient_accumulation_steps": 1,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "logging_steps": 50,
    "save_total_limit": 2,
    "report_to": "none",
}

training_args = TrainingArguments(
    output_dir="./mnbert-4class-checkpoints",
    **CONFIG,
)

print("Training config:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 7. Compute Metrics

In [ ]:
LABEL_NAMES = [ID2LABEL[i] for i in range(NUM_LABELS)]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_mac = f1_score(labels, preds, average="macro")
    f1_wt = f1_score(labels, preds, average="weighted")

    precision, recall, f1_per, _ = precision_recall_fscore_support(
        labels, preds, average=None, labels=list(range(NUM_LABELS))
    )

    metrics = {
        "accuracy": acc,
        "f1_macro": f1_mac,
        "f1_weighted": f1_wt,
    }
    for i, name in enumerate(LABEL_NAMES):
        metrics[f"precision_{name}"] = precision[i]
        metrics[f"recall_{name}"] = recall[i]
        metrics[f"f1_{name}"] = f1_per[i]

    return metrics

## 8. Train

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_dict["train"],
    eval_dataset=ds_dict["val"],
    compute_metrics=compute_metrics,
)

start_time = time.time()
trainer.train()
duration = time.time() - start_time

mins, secs = divmod(int(duration), 60)
print(f"\nTraining completed in {mins}m {secs}s")

## 9. Evaluate on Test Split

In [ ]:
preds_output = trainer.predict(ds_dict["test"])
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

print("=" * 60)
print("CLASSIFICATION REPORT (Test Set)")
print("=" * 60)
print(classification_report(labels, preds, target_names=LABEL_NAMES, digits=4))

print("\nCONFUSION MATRIX")
print(f"{'':>14}", "  ".join(f"{n:>12}" for n in LABEL_NAMES))
cm = confusion_matrix(labels, preds, labels=list(range(NUM_LABELS)))
for i, row in enumerate(cm):
    print(f"{LABEL_NAMES[i]:>14}", "  ".join(f"{v:>12}" for v in row))

## 10. Save Model & Tokenizer

In [ ]:
SAVE_DIR = "./mnbert-4class-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model and tokenizer saved to {SAVE_DIR}")